# UK surface transport emissions: solutions

Model answers for the post-training exercises. See `exercises.ipynb` for the environment setup and a description of the data.

In [ ]:
import os

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Q1) Get to know the data

In [ ]:
# Q1a)
cars = pd.read_csv("Cars.csv")
print(cars.head())

# Q1b)
print(cars.shape)

# Q1c)
print(cars["series"].unique())

## Q2) Filtering, conditionals and loops

In [ ]:
# Q2a)
value = 58

if value > 40:
    print("high")
elif value > 20:
    print("medium")
else:
    print("low")

In [ ]:
# Q2b)
pathway = cars.loc[cars["series"] == "Balanced Pathway"]
print(pathway.shape[0], "years")

# Q2c)
for year, value in zip(pathway["year"], pathway["emissions"]):
    print(year, value)

In [ ]:
# Q2d) - filter once, then loop with a condition inside
for year, value in zip(pathway["year"], pathway["emissions"]):
    if value < 10:
        print(year, value)

# Q2e) - the same years in one step, with two conditions in one mask
low_years = cars.loc[
    (cars["series"] == "Balanced Pathway") & (cars["emissions"] < 10)
]
print()
print(low_years)

In [ ]:
# Q2f)
value = 57.8
years = 0

while value > 10:
    value = value * 0.9
    years += 1

print(years, "years of steady 10% cuts, so below 10 MtCO2e by", 2025 + years)
print("emissions then:", round(value, 2))

# A steady 10% a year gets below 10 MtCO2e in 2042. The Balanced Pathway does it
# in 2039, three years sooner.

## Q3) Your first plot

In [ ]:
# Q3a)
fig, ax = plt.subplots()
ax.plot(pathway["year"], pathway["emissions"])

# Q3b)
ax.set_xlabel("Year")
ax.set_ylabel("Emissions / MtCO2e")
ax.set_title("UK car emissions, Balanced Pathway")

# Q3c)
ax.set_xticks([2030, 2040, 2050])

# Q3d)
fig.savefig("car_emissions.png")

## Q4) The CCC theme

In [ ]:
# Q4a) - importing the theme is what applies it
import ccc_theme as ccc

fig, ax = plt.subplots()
ax.plot(pathway["year"], pathway["emissions"])

ax.set_xlabel("Year")
ax.set_ylabel("Emissions / MtCO2e")
ax.set_title("UK car emissions, Balanced Pathway")

In [ ]:
# Q4b)
historical = cars.loc[cars["series"] == "Historical"]

historical_colour = ccc.SCENARIO_COLORS["Historical"]
pathway_colour = ccc.SCENARIO_COLORS["Pathway"]

fig, ax = plt.subplots()

ax.plot(historical["year"], historical["emissions"], color=historical_colour, label="Historical")
ax.plot(pathway["year"], pathway["emissions"], color=pathway_colour, label="Balanced Pathway")

ax.set_xlabel("Year")
ax.set_ylabel("Emissions / MtCO2e")
ax.legend()

# Q4c) - one small change on top of the theme
ax.set_title("UK car emissions", fontdict={"fontsize": 15})

fig.savefig("car_emissions_ccc.png")

## Q5) Two panels, and a trend line

In [ ]:
# Q5a)
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(9, 4))

axs[0].plot(historical["year"], historical["emissions"], color=historical_colour)
axs[1].plot(pathway["year"], pathway["emissions"], color=pathway_colour)

# Q5b)
axs[0].set_title("Historical")
axs[1].set_title("Balanced Pathway")

for ax in axs:
    ax.set_xlabel("Year")

axs[0].set_ylabel("Emissions / MtCO2e")

In [ ]:
# Q5c)
sns.regplot(data=pathway, x="year", y="emissions")
plt.show()

# Q5d) - the same plot, drawn onto axes we control
fig, ax = plt.subplots(figsize=(5, 4))

sns.regplot(data=pathway, x="year", y="emissions", ax=ax)

ax.set_xlabel("Year")
ax.set_ylabel("Emissions / MtCO2e")
ax.set_title("A straight line is a poor fit for the pathway")

plt.show()

## Q6) Five-year budget periods

In [ ]:
# Q6a)
print(pathway["emissions"].sum())

# Q6b)
first_period = pathway.loc[
    (pathway["year"] >= 2025) & (pathway["year"] <= 2029)
]
print(first_period["emissions"].sum())

In [ ]:
# Q6c)
starts = [2025, 2030, 2035, 2040, 2045]

period_totals = []
for start in starts:
    period = pathway.loc[
        (pathway["year"] >= start) & (pathway["year"] <= start + 4)
    ]
    period_totals.append(period["emissions"].sum())

# Printing with zip() keeps each total next to the period it belongs to
for start, total in zip(starts, period_totals):
    print(start, round(total, 1))

In [ ]:
# Q6d)
labels = ["2025-29", "2030-34", "2035-39", "2040-44", "2045-49"]

fig, ax = plt.subplots()

ax.bar(labels, period_totals)

ax.set_xlabel("Carbon budget period")
ax.set_ylabel("Emissions / MtCO2e")
ax.set_title("Car emissions per five-year budget period")

fig.savefig("car_budget_periods.png")

## Q7) Write a function

In [ ]:
# Q7a), Q7c) and Q7d) - the finished function
def plot_subsector(subsector, series="Balanced Pathway", ax=None):
    """
    Plot one series of one subsector's emissions over time.

    Parameters
    ----------
    subsector : str
        Subsector name, matching its csv file, eg. "Vans".
    series : str
        Which series to plot, eg. "Historical". Defaults to "Balanced Pathway".
    ax : matplotlib.axes.Axes, optional
        Axes to draw on. A new figure is made if this is not given.

    Returns
    -------
    matplotlib.axes.Axes
        The axes the chart was drawn on, so that the caller can carry on
        customising it.
    """
    data = pd.read_csv(subsector + ".csv")
    rows = data.loc[data["series"] == series]

    if ax is None:
        fig, ax = plt.subplots()

    ax.plot(rows["year"], rows["emissions"])
    ax.set_xlabel("Year")
    ax.set_ylabel("Emissions / MtCO2e")
    ax.set_title(subsector)

    return ax


# Q7b)
plot_subsector("Vans")
plot_subsector("HGVs")

# Q7c) - the default can be overridden to plot the historical years instead
plot_subsector("Cars", "Historical")

plt.show()

## Q8) Every subsector at once

In [ ]:
# Q8a)
files = os.listdir()
print(files)

# Q8b)
for file in files:
    if file.endswith(".csv"):
        print(file)

In [ ]:
# Q8c)
vans = pd.read_csv("Vans.csv")
vans["subsector"] = "Vans"

print(vans.head())

In [ ]:
# Q8d)
emissions = pd.DataFrame()

for file in files:
    if file.endswith(".csv"):
        subsector_data = pd.read_csv(file)
        subsector_data["subsector"] = file.replace(".csv", "")
        emissions = pd.concat([emissions, subsector_data])

# Q8e)
print(emissions.shape)
emissions.head()

## Q9) Comparing the subsectors

In [ ]:
# Q9a) - one line in the loop, because plot_subsector() takes an ax
subsectors = emissions["subsector"].unique()

fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(12, 6))

for ax, subsector in zip(axs.flatten(), subsectors):
    plot_subsector(subsector, ax=ax)

# Five subsectors, six panels
axs.flatten()[-1].set_visible(False)

fig.savefig("subsector_panels.png")

In [ ]:
# Q9b)
subsector_totals = []
for subsector in subsectors:
    rows = emissions.loc[
        (emissions["subsector"] == subsector)
        & (emissions["series"] == "Balanced Pathway")
    ]
    subsector_totals.append(rows["emissions"].sum())

for subsector, total in zip(subsectors, subsector_totals):
    print(subsector, round(total, 1))

fig, ax = plt.subplots()

ax.bar(subsectors, subsector_totals)

ax.set_xlabel("Subsector")
ax.set_ylabel("Emissions 2025-2050 / MtCO2e")
ax.set_title("Cars dominate the surface transport budget")

fig.savefig("subsector_totals.png")

## Q10) Move your code into a script

The answer here is a file rather than a cell. Your `transport_tools.py` should look something like this:

```python
import pandas as pd
import matplotlib.pyplot as plt


def plot_subsector(subsector, series="Balanced Pathway", ax=None):
    """
    Plot one series of one subsector's emissions over time.

    Parameters
    ----------
    subsector : str
        Subsector name, matching its csv file, eg. "Vans".
    series : str
        Which series to plot, eg. "Historical". Defaults to "Balanced Pathway".
    ax : matplotlib.axes.Axes, optional
        Axes to draw on. A new figure is made if this is not given.

    Returns
    -------
    matplotlib.axes.Axes
        The axes the chart was drawn on, so that the caller can carry on
        customising it.
    """
    data = pd.read_csv(subsector + ".csv")
    rows = data.loc[data["series"] == series]

    if ax is None:
        fig, ax = plt.subplots()

    ax.plot(rows["year"], rows["emissions"])
    ax.set_xlabel("Year")
    ax.set_ylabel("Emissions / MtCO2e")
    ax.set_title(subsector)

    return ax


if __name__ == "__main__":
    for subsector in ["Cars", "Vans", "HGVs", "Rail", "Other"]:
        ax = plot_subsector(subsector)
        ax.figure.savefig(subsector + ".png")
        print("wrote", subsector + ".png")
```

Both `pandas` and `matplotlib.pyplot` are imported at the top, because the function needs both. A script has to stand on its own - it can't rely on the imports you happen to have run in a notebook.

`%run transport_tools.py` writes the five figures, because running a script executes the `__main__` block. `import transport_tools` writes nothing, because importing it does not - which is the whole point of the guard. It means the same file works both as a thing to run and as a thing to import.

In [ ]:
# Q10c) - Q10e), once transport_tools.py exists:
#
#     from transport_tools import plot_subsector
#     plot_subsector("Vans")
#
#     %run transport_tools.py     # writes the five figures
#     import transport_tools      # writes nothing